# 🎯 Scoring Profile 실습

Azure AI Search의 Scoring Profile을 활용하여 검색 결과 순위에 비즈니스 로직을 적용하는 방법을 학습합니다.

## 학습 목표
1. Scoring Profile의 개념과 구성 요소 이해
2. 필드 가중치(Field Weights) 설정
3. Magnitude 함수로 가격 기반 부스팅
4. Tag 함수로 특정 브랜드 우선 노출
5. 프로파일 적용 전/후 결과 비교

---
## 1️⃣ 환경 설정

In [6]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    ScoringProfile,
    TextWeights,
    ScoringFunction,
    MagnitudeScoringFunction,
    MagnitudeScoringParameters,
    TagScoringFunction,
    TagScoringParameters,
    ScoringFunctionInterpolation,
)

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")

credential = AzureKeyCredential(SEARCH_ADMIN_KEY)

print(f"✅ Endpoint: {SEARCH_ENDPOINT}")
print(f"✅ Index: {INDEX_NAME}")

✅ Endpoint: https://foundryiq-aisearch-260114-changjuahn.search.windows.net
✅ Index: products-index


---
## 2️⃣ Scoring Profile 개념 이해

### Scoring Profile이란?

**Scoring Profile**은 검색 결과의 순위를 비즈니스 요구사항에 맞게 조정하는 기능입니다.

```
최종 점수 = BM25 기본 점수 × 필드 가중치 × 함수 부스팅
```

### 주요 구성 요소

| 구성 요소 | 설명 | 적용 예시 |
|----------|------|----------|
| **Text Weights** | 필드별 가중치 | title 매칭 시 2배 점수 |
| **Magnitude** | 숫자 필드 기반 | 저가 상품 우선 |
| **Tag** | 특정 값 매칭 | 프리미엄 브랜드 우선 |
| **Freshness** | 날짜 기반 | 최신 상품 우선 |
| **Distance** | 위치 기반 | 가까운 매장 우선 |

---
## 3️⃣ 기존 인덱스에 Scoring Profile 추가

이미 02-keyword_search, 03-vector_search 실습에서 생성한 인덱스에 Scoring Profile을 추가합니다.

In [7]:
# 인덱스 클라이언트 초기화
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

# ========================================
# Scoring Profile 정의
# ========================================

# 프로파일 1: 필드 가중치 (title 매칭 우선)
profile_field_weights = ScoringProfile(
    name="titleBoost",
    text_weights=TextWeights(
        weights={
            "title": 3.0,      # title 필드 매칭 시 3배
            "brand": 2.0,      # brand 필드 매칭 시 2배
            "review": 1.0      # review 필드는 기본
        }
    )
)

# 프로파일 2: 저가 상품 우선 (Magnitude 함수)
profile_low_price = ScoringProfile(
    name="lowPriceFirst",
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=5,  # 부스팅 배수
            parameters=MagnitudeScoringParameters(
                boosting_range_start=0,       # 0원부터
                boosting_range_end=100000,    # 10만원까지
                constant_boost_beyond_range=False  # 범위 밖은 부스팅 감소
            ),
            interpolation=ScoringFunctionInterpolation.LINEAR  # 선형 감소
        )
    ]
)

# 프로파일 3: 고가 상품 우선 (프리미엄 전략)
profile_high_price = ScoringProfile(
    name="highPriceFirst",
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=5,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=100000,   # 10만원부터
                boosting_range_end=500000,     # 50만원까지
                constant_boost_beyond_range=True  # 범위 밖도 최대 부스팅 유지
            ),
            interpolation=ScoringFunctionInterpolation.LINEAR
        )
    ]
)

# 프로파일 4: 프리미엄 브랜드 우선 (Tag 함수)
profile_premium_brand = ScoringProfile(
    name="premiumBrandFirst",
    functions=[
        TagScoringFunction(
            field_name="brand",
            boost=10,  # 매칭 시 10배 부스팅
            parameters=TagScoringParameters(
                tags_parameter="premiumBrands"  # 쿼리 시 전달할 파라미터 이름
            )
        )
    ]
)

# 프로파일 5: 복합 프로파일 (필드 가중치 + 저가 우선)
profile_combined = ScoringProfile(
    name="bestValue",
    text_weights=TextWeights(
        weights={
            "title": 2.0,
            "brand": 1.5
        }
    ),
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=3,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=0,
                boosting_range_end=80000,
                constant_boost_beyond_range=False
            ),
            interpolation=ScoringFunctionInterpolation.QUADRATIC  # 2차 함수 감소
        )
    ],
    function_aggregation="sum"  # 여러 함수 결과 합산
)

print("✅ 5개의 Scoring Profile 정의 완료")
print("   1. titleBoost: 필드 가중치 (title 3배, brand 2배)")
print("   2. lowPriceFirst: 저가 상품 우선")
print("   3. highPriceFirst: 고가 상품 우선")
print("   4. premiumBrandFirst: 프리미엄 브랜드 우선")
print("   5. bestValue: 복합 (필드 가중치 + 저가 우선)")

constant_boost_beyond_range is not a known attribute of class <class 'azure.search.documents.indexes._generated.models._models_py3.MagnitudeScoringParameters'> and will be ignored
constant_boost_beyond_range is not a known attribute of class <class 'azure.search.documents.indexes._generated.models._models_py3.MagnitudeScoringParameters'> and will be ignored
constant_boost_beyond_range is not a known attribute of class <class 'azure.search.documents.indexes._generated.models._models_py3.MagnitudeScoringParameters'> and will be ignored


✅ 5개의 Scoring Profile 정의 완료
   1. titleBoost: 필드 가중치 (title 3배, brand 2배)
   2. lowPriceFirst: 저가 상품 우선
   3. highPriceFirst: 고가 상품 우선
   4. premiumBrandFirst: 프리미엄 브랜드 우선
   5. bestValue: 복합 (필드 가중치 + 저가 우선)


In [8]:
# 기존 인덱스 가져오기
existing_index = index_client.get_index(INDEX_NAME)

print(f"✅ 기존 인덱스 '{INDEX_NAME}' 로드 완료")
print(f"   - 필드 수: {len(existing_index.fields)}")

# 기존 Scoring Profiles에 새 프로파일 추가
existing_profiles = list(existing_index.scoring_profiles) if existing_index.scoring_profiles else []

# 중복 제거 (이미 있으면 교체)
new_profiles = [
    profile_field_weights,
    profile_low_price,
    profile_high_price,
    profile_premium_brand,
    profile_combined
]

for new_profile in new_profiles:
    existing_profiles = [p for p in existing_profiles if p.name != new_profile.name]
    existing_profiles.append(new_profile)

# 인덱스에 Scoring Profiles 추가
existing_index.scoring_profiles = existing_profiles

# 인덱스 업데이트
result = index_client.create_or_update_index(existing_index)

print(f"\n✅ Scoring Profile 추가 완료!")
print(f"   - 총 Scoring Profiles: {len(result.scoring_profiles)}개")
for profile in result.scoring_profiles:
    print(f"     • {profile.name}")

✅ 기존 인덱스 'products-index' 로드 완료
   - 필드 수: 8

✅ Scoring Profile 추가 완료!
   - 총 Scoring Profiles: 5개
     • titleBoost
     • lowPriceFirst
     • highPriceFirst
     • premiumBrandFirst
     • bestValue


---
## 4️⃣ 기존 데이터 확인

In [9]:
# 검색 클라이언트 초기화
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

# 기존 데이터 확인
result = search_client.search(search_text="*", top=1, include_total_count=True)
doc_count = result.get_count()

print(f"✅ 기존 인덱스 '{INDEX_NAME}'에 {doc_count}개의 문서가 존재합니다.")
print(f"   → 이전 실습(02-keyword_search, 03-vector_search)에서 업로드한 데이터를 사용합니다.")

✅ 기존 인덱스 'products-index'에 247개의 문서가 존재합니다.
   → 이전 실습(02-keyword_search, 03-vector_search)에서 업로드한 데이터를 사용합니다.


---
## 5️⃣ Scoring Profile 비교 실습

### 실습 1: 프로파일 미적용 vs 필드 가중치 적용

In [10]:
def search_with_profile(query, profile_name=None, top=5):
    """Scoring Profile을 적용한 검색"""
    results = search_client.search(
        search_text=query,
        scoring_profile=profile_name,
        top=top,
        include_total_count=True
    )
    
    profile_text = profile_name if profile_name else "없음(기본)"
    print(f"\n🔍 검색어: '{query}'")
    print(f"📊 Scoring Profile: {profile_text}")
    print("=" * 80)
    
    results_list = []
    for idx, result in enumerate(results, 1):
        print(f"{idx}. {result['title'][:50]}...")
        print(f"   브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
        print(f"   점수: {result['@search.score']:.4f}")
        results_list.append({
            'title': result['title'],
            'brand': result['brand'],
            'price': result['normal_price'],
            'score': result['@search.score']
        })
    
    return results_list

In [11]:
# 비교 1: "선물" 검색 - 기본 vs titleBoost
print("\n" + "="*80)
print(" 비교 1: 필드 가중치 효과 (titleBoost)")
print("="*80)

# 기본 검색 (프로파일 없음)
results_default = search_with_profile("선물", profile_name=None, top=5)

# titleBoost 프로파일 적용
results_title = search_with_profile("선물", profile_name="titleBoost", top=5)


 비교 1: 필드 가중치 효과 (titleBoost)

🔍 검색어: '선물'
📊 Scoring Profile: 없음(기본)
1. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 5.8588
2. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형...
   브랜드: 블루독베이비 | 가격: 36,000원
   점수: 5.2385
3. 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   브랜드: 압소바 | 가격: 39,000원
   점수: 4.6017
4. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출...
   브랜드: 에뜨와 | 가격: 98,000원
   점수: 4.3339
5. 사브르 비스트로 커트러리 빈티지 디너 선물세트...
   브랜드: 사브르 | 가격: 50,160원
   점수: 4.2787

🔍 검색어: '선물'
📊 Scoring Profile: titleBoost
1. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형...
   브랜드: 블루독베이비 | 가격: 36,000원
   점수: 11.8577
2. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 10.9179
3. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출...
   브랜드: 에뜨와 | 가격: 98,000원
   점수: 10.0947
4. [노베즈][선물포장] 촉촉3종세트 (로션+워시+크림) 선물추천 스킨케어선물 화장품세트...
   브랜드: 노베즈 | 가격: 81,000원
   점수: 8.7390
5. [멜리멜로] 생일/집들이선물 클래식 스톤 디퓨저 생일 집들이 친구선물...
   브랜드: 멜리멜로 | 가격: 28,000원
   점수: 8.4638


In [19]:
# 점수 비교 시각화
print("\n📈 점수 비교 (기본 vs titleBoost)")
print("-" * 60)
print(f"{'순위':<5} {'기본 점수':<10} {'titleBoost 점수':<17} {'차이'}")
print("-" * 60)
for i, (d, t) in enumerate(zip(results_default, results_title), 1):
    diff = t['score'] - d['score']
    diff_text = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    print(f"{i:<5} {d['score']:<15.4f} {t['score']:<15.4f} {diff_text}")


📈 점수 비교 (기본 vs titleBoost)
------------------------------------------------------------
순위    기본 점수      titleBoost 점수     차이
------------------------------------------------------------
1     5.8588          11.8577         +5.9989
2     5.2385          10.9179         +5.6793
3     4.6017          10.0947         +5.4930
4     4.3339          8.7390          +4.4051
5     4.2787          8.4638          +4.1851


### 실습 2: 저가 우선 vs 고가 우선 비교

In [20]:
print("\n" + "="*80)
print(" 비교 2: 가격 기반 부스팅 (저가 vs 고가)")
print("="*80)

# 저가 우선
results_low = search_with_profile("자켓", profile_name="lowPriceFirst", top=5)

# 고가 우선
results_high = search_with_profile("자켓", profile_name="highPriceFirst", top=5)


 비교 2: 가격 기반 부스팅 (저가 vs 고가)

🔍 검색어: '자켓'
📊 Scoring Profile: lowPriceFirst
1. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 65,550원
   점수: 20.8368
2. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 6.2902
3. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 5.8492
4. 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HCZ...
   브랜드: 라코스테 우먼 | 가격: 213,570원
   점수: 3.7855

🔍 검색어: '자켓'
📊 Scoring Profile: highPriceFirst
1. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 9.8001
2. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 9.1131
3. 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HCZ...
   브랜드: 라코스테 우먼 | 가격: 213,570원
   점수: 8.0846
4. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 65,550원
   점수: 5.7529


In [21]:
# 가격 순서 비교
print("\n💰 가격 비교")
print("-" * 80)
print(f"{'순위':<5} {'lowPriceFirst':<25} {'가격':<15} {'highPriceFirst':<25} {'가격'}")
print("-" * 80)
for i, (low, high) in enumerate(zip(results_low, results_high), 1):
    low_title = low['title'][:20] + "..."
    high_title = high['title'][:20] + "..."
    print(f"{i:<5} {low_title:<25} {low['price']:>10,}원   {high_title:<25} {high['price']:>10,}원")


💰 가격 비교
--------------------------------------------------------------------------------
순위    lowPriceFirst             가격              highPriceFirst            가격
--------------------------------------------------------------------------------
1     노스페이스 NJ3LR05 남성 아이스...       65,550원   노스페이스 NJ2HR11 남성 패커블...      155,800원
2     노스페이스 NJ2HR11 남성 패커블...      155,800원   노스페이스 NJ2HR41 여성 패커블...      155,800원
3     노스페이스 NJ2HR41 여성 패커블...      155,800원   라코스테우먼 25SS 여성 오버사이즈...      213,570원
4     라코스테우먼 25SS 여성 오버사이즈...      213,570원   노스페이스 NJ3LR05 남성 아이스...       65,550원


### 실습 3: Tag 함수를 활용한 프리미엄 브랜드 부스팅

In [22]:
print("\n" + "="*80)
print(" 비교 3: 프리미엄 브랜드 부스팅 (Tag 함수)")
print("="*80)

# Tag 함수 사용을 위한 검색
# premiumBrands 파라미터에 부스팅할 브랜드 목록 전달
premium_brands = ["노스페이스", "라코스테 우먼", "앤더슨벨"]

# 기본 검색
print("\n📌 기본 검색 (프로파일 없음)")
results_no_tag = search_with_profile("자켓", profile_name=None, top=5)

# Tag 함수 적용 검색
print(f"\n📌 프리미엄 브랜드 우선 (premiumBrands: {premium_brands})")
results_with_tag = search_client.search(
    search_text="자켓",
    scoring_profile="premiumBrandFirst",
    scoring_parameters=[f"premiumBrands-{','.join(premium_brands)}"],
    top=5
)

print("\n🔍 검색어: '자켓'")
print("📊 Scoring Profile: premiumBrandFirst")
print(f"🏷️ 부스팅 브랜드: {premium_brands}")
print("=" * 80)

for idx, result in enumerate(results_with_tag, 1):
    is_premium = "⭐" if result['brand'] in premium_brands else "  "
    print(f"{idx}. {is_premium} {result['title'][:45]}...")
    print(f"      브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
    print(f"      점수: {result['@search.score']:.4f}")


 비교 3: 프리미엄 브랜드 부스팅 (Tag 함수)

📌 기본 검색 (프로파일 없음)

🔍 검색어: '자켓'
📊 Scoring Profile: 없음(기본)
1. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 6.2902
2. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 가격: 155,800원
   점수: 5.8492
3. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 가격: 65,550원
   점수: 5.7529
4. 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HCZ...
   브랜드: 라코스테 우먼 | 가격: 213,570원
   점수: 3.7855

📌 프리미엄 브랜드 우선 (premiumBrands: ['노스페이스', '라코스테 우먼', '앤더슨벨'])

🔍 검색어: '자켓'
📊 Scoring Profile: premiumBrandFirst
🏷️ 부스팅 브랜드: ['노스페이스', '라코스테 우먼', '앤더슨벨']
1. ⭐ 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
      브랜드: 노스페이스 | 가격: 155,800원
      점수: 62.9018
2. ⭐ 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
      브랜드: 노스페이스 | 가격: 155,800원
      점수: 58.4920
3. ⭐ 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
      브랜드: 노스페이스 | 가격: 65,550원
      점수: 57.5285
4. ⭐ 라코스테우먼 25SS 여성 오버사이즈 컬러데님 셔츠 자켓 BF2446-55G HC...
      브랜드: 라코스테 우먼 | 가격: 213,

### 실습 4: 복합 프로파일 (bestValue)

In [23]:
print("\n" + "="*80)
print(" 비교 4: 복합 프로파일 (필드 가중치 + 저가 우선)")
print("="*80)

# 기본 검색
results_basic = search_with_profile("유아용품 선물", profile_name=None, top=5)

# 복합 프로파일 적용
results_best = search_with_profile("유아용품 선물", profile_name="bestValue", top=5)


 비교 4: 복합 프로파일 (필드 가중치 + 저가 우선)

🔍 검색어: '유아용품 선물'
📊 Scoring Profile: 없음(기본)
1. [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카...
   브랜드: 밍크뮤 | 가격: 35,280원
   점수: 8.6405
2. 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건...
   브랜드: 블루독베이비 | 가격: 22,190원
   점수: 7.2922
3. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출...
   브랜드: 에뜨와 | 가격: 98,000원
   점수: 6.7053
4. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 5.8588
5. (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/...
   브랜드: 에뜨와 | 가격: 38,000원
   점수: 5.7729

🔍 검색어: '유아용품 선물'
📊 Scoring Profile: bestValue
1. [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카...
   브랜드: 밍크뮤 | 가격: 35,280원
   점수: 21.8218
2. (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/...
   브랜드: 에뜨와 | 가격: 38,000원
   점수: 14.3405
3. 사브르 비스트로 샤이니 샐러드 선물세트...
   브랜드: 사브르 | 가격: 47,120원
   점수: 14.2085
4. 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(35W70-BHB-0...
   브랜드: 밍크뮤 | 가격: 75,200원
   점수: 14.1826
5. 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건.

---
## 6️⃣ Interpolation 방식 비교

Magnitude 함수에서 점수가 어떻게 감소하는지 보간 방식을 시각화합니다.

In [24]:
import math

# Interpolation 방식별 점수 계산 함수
def calculate_boost(value, start, end, boost, interpolation):
    """부스팅 점수 계산 (Azure AI Search 방식 시뮬레이션)"""
    if value <= start:
        return boost
    elif value >= end:
        return 1  # 기본 점수
    
    # 정규화된 위치 (0~1)
    position = (value - start) / (end - start)
    
    if interpolation == "linear":
        return boost - (boost - 1) * position
    elif interpolation == "quadratic":
        return boost - (boost - 1) * (position ** 2)
    elif interpolation == "logarithmic":
        return boost - (boost - 1) * math.log10(1 + 9 * position)
    elif interpolation == "constant":
        return boost

# 가격 범위에 따른 부스팅 점수 시각화 (텍스트)
print("\n📊 Interpolation 방식별 부스팅 점수 비교")
print("   (가격 범위: 0원 ~ 100,000원, 부스팅 배수: 5)")
print("=" * 75)
print(f"{'가격':<12} {'Linear':<12} {'Quadratic':<12} {'Logarithmic':<12} {'Constant'}")
print("-" * 75)

prices = [0, 20000, 40000, 60000, 80000, 100000, 150000]
for price in prices:
    linear = calculate_boost(price, 0, 100000, 5, "linear")
    quadratic = calculate_boost(price, 0, 100000, 5, "quadratic")
    logarithmic = calculate_boost(price, 0, 100000, 5, "logarithmic")
    constant = calculate_boost(price, 0, 100000, 5, "constant")
    
    print(f"{price:>10,}원 {linear:<12.2f} {quadratic:<12.2f} {logarithmic:<12.2f} {constant:.2f}")


📊 Interpolation 방식별 부스팅 점수 비교
   (가격 범위: 0원 ~ 100,000원, 부스팅 배수: 5)
가격           Linear       Quadratic    Logarithmic  Constant
---------------------------------------------------------------------------
         0원 5.00         5.00         5.00         5.00
    20,000원 4.20         4.84         3.21         5.00
    40,000원 3.40         4.36         2.35         5.00
    60,000원 2.60         3.56         1.78         5.00
    80,000원 1.80         2.44         1.34         5.00
   100,000원 1.00         1.00         1.00         1.00
   150,000원 1.00         1.00         1.00         1.00


### Interpolation 방식 설명

| 방식 | 특징 | 사용 케이스 |
|------|------|------------|
| **Linear** | 균일하게 감소 | 일반적인 가격 부스팅 |
| **Quadratic** | 처음에는 천천히, 끝에서 급격히 감소 | 저가 구간 강조 |
| **Logarithmic** | 처음에 급격히, 끝에서 천천히 감소 | 중간 가격대 강조 |
| **Constant** | 범위 내 동일 부스팅 | 특정 범위 전체 강조 |

---
## 7️⃣ 실무 시나리오: 카테고리별 맞춤 검색

In [25]:
print("\n" + "="*80)
print(" 실무 시나리오: 카테고리별 맞춤 검색")
print("="*80)

# 시나리오 1: 유아동 - 가성비 상품 우선
print("\n📍 시나리오 1: 유아동 카테고리 - 가성비 상품 추천")
print("   → lowPriceFirst 프로파일 + category 필터")

results = search_client.search(
    search_text="선물",
    filter="category eq '유아동'",
    scoring_profile="lowPriceFirst",
    top=5
)

print("\n🔍 검색어: '선물' (유아동 카테고리)")
print("-" * 60)
for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['title'][:40]}...")
    print(f"   💰 가격: {result['normal_price']:,}원 | 점수: {result['@search.score']:.4f}")


 실무 시나리오: 카테고리별 맞춤 검색

📍 시나리오 1: 유아동 카테고리 - 가성비 상품 추천
   → lowPriceFirst 프로파일 + category 필터

🔍 검색어: '선물' (유아동 카테고리)
------------------------------------------------------------
1. (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S717...
   💰 가격: 98,000원 | 점수: 21.3227
2. [노베즈][선물포장] 촉촉3종세트 (로션+워시+크림) 선물추천 스킨케어선...
   💰 가격: 81,000원 | 점수: 16.2801
3. 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(3...
   💰 가격: 75,200원 | 점수: 13.0758
4. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02...
   💰 가격: 36,000원 | 점수: 12.7820
5. [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임...
   💰 가격: 96,000원 | 점수: 12.5074


In [26]:
# 시나리오 2: 스포츠/레져 - 프리미엄 브랜드 우선
print("\n📍 시나리오 2: 스포츠/레져 카테고리 - 프리미엄 브랜드 추천")
print("   → premiumBrandFirst 프로파일 + category 필터")

sports_premium = ["노스페이스", "파타고니아", "젝시믹스"]

results = search_client.search(
    search_text="자켓 바람막이",
    filter="category eq '스포츠/레져'",
    scoring_profile="premiumBrandFirst",
    scoring_parameters=[f"premiumBrands-{','.join(sports_premium)}"],
    top=5
)

print(f"\n🔍 검색어: '자켓 바람막이' (스포츠/레져 카테고리)")
print(f"🏷️ 프리미엄 브랜드: {sports_premium}")
print("-" * 60)
for idx, result in enumerate(results, 1):
    is_premium = "⭐" if result['brand'] in sports_premium else "  "
    print(f"{idx}. {is_premium} {result['title'][:40]}...")
    print(f"      브랜드: {result['brand']} | 점수: {result['@search.score']:.4f}")


📍 시나리오 2: 스포츠/레져 카테고리 - 프리미엄 브랜드 추천
   → premiumBrandFirst 프로파일 + category 필터

🔍 검색어: '자켓 바람막이' (스포츠/레져 카테고리)
🏷️ 프리미엄 브랜드: ['노스페이스', '파타고니아', '젝시믹스']
------------------------------------------------------------
1. ⭐ 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자...
      브랜드: 노스페이스 | 점수: 220.2112
2. ⭐ 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자...
      브랜드: 노스페이스 | 점수: 213.6705
3. ⭐ 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 ...
      브랜드: 노스페이스 | 점수: 210.1245


---
## 9️⃣ 정리 및 핵심 포인트

### ✅ 학습한 내용

1. **Scoring Profile 기본 개념**
   - BM25 기본 점수에 비즈니스 로직 추가
   - 인덱스 정의 시 포함, 쿼리 시 선택 적용

2. **필드 가중치 (Text Weights)**
   - `title`, `brand` 등 필드별 중요도 설정
   - 상품명 매칭 시 높은 점수 부여

3. **Magnitude 함수**
   - 숫자 필드(가격) 기반 부스팅
   - `boostingRangeStart/End`로 범위 설정
   - Interpolation: Linear, Quadratic, Logarithmic, Constant

4. **Tag 함수**
   - 특정 값(브랜드, 카테고리) 매칭 시 부스팅
   - 쿼리 시 `scoring_parameters`로 값 전달

### ⚠️ 주의사항

- Scoring Profile은 **키워드 검색(BM25)에만 적용**
- 벡터 검색, 하이브리드 검색의 RRF 점수에는 영향 없음
- 프로파일 변경 시 **인덱스 업데이트 필요**

### 🎯 실무 활용 팁

| 비즈니스 목표 | 권장 프로파일 |
|--------------|---------------|
| 가성비 상품 추천 | Magnitude (저가 우선) |
| 프리미엄 상품 추천 | Tag (특정 브랜드) |
| 검색 정확도 향상 | Text Weights (title 강조) |
| 파트너사 상품 노출 | Tag (파트너 브랜드) |

In [ ]:
# (참고) Scoring Profile 제거 방법
# 특정 프로파일을 제거하려면:
# existing_index = index_client.get_index(INDEX_NAME)
# existing_index.scoring_profiles = [p for p in existing_index.scoring_profiles if p.name not in ['titleBoost', 'lowPriceFirst']]
# index_client.create_or_update_index(existing_index)
# print("✅ Scoring Profile 제거 완료")

print("💡 Scoring Profile은 인덱스에 영구적으로 저장됩니다.")
print("💡 제거하려면 위의 주석 코드를 참고하세요.")